# Encrypted Search Researcher Tutorial

This notebook demonstrates the encrypted-search research workflow in SecureRAG.

You will:
- Build encrypted corpora using SSE and Structured schemes.
- Run retrieval through the unified `encrypted_search` runtime path.
- Inspect encrypted query artifacts without exposing plaintext terms on server-side flow.
- Validate version-guard behavior for legacy encrypted indexes.

In [ ]:
from __future__ import annotations

import socket
import subprocess
import sys
import time

import httpx

from securerag import PrivacyConfig, PrivacyProtocol, SecureRAGAgent
from securerag.backend_client import create_backend
from securerag.corpus import CorpusBuilder
from securerag.errors import BackendError
from securerag.llm import OllamaLLM
from securerag.models import RawDocument
from securerag.scheme_plugin import EncryptedSchemePlugin


def free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])


def wait_for_health(base_url: str, timeout: float = 10.0) -> None:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            r = httpx.get(f"{base_url}/health", timeout=0.4)
            if r.status_code == 200:
                return
        except Exception:
            pass
        time.sleep(0.1)
    raise RuntimeError(f"sim_server did not become healthy at {base_url}")

In [ ]:
port = free_port()
base_url = f"http://127.0.0.1:{port}"
sim_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "securerag.sim_server:app",
        "--host",
        "127.0.0.1",
        "--port",
        str(port),
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
wait_for_health(base_url)

docs = [
    RawDocument(doc_id="q3", text="Q3 risk report highlights vendor concentration and delayed remediation."),
    RawDocument(doc_id="policy", text="Security policy mandates quarterly risk treatment and owner assignment."),
    RawDocument(doc_id="ops", text="Incident patterns suggest ingestion queue saturation under peak traffic."),
]

print("sim_server ready at", base_url)

## 1) Build and Query with SSE Scheme

In [ ]:
sse_corpus = (
    CorpusBuilder(PrivacyProtocol.ENCRYPTED_SEARCH, backend_url=base_url)
    .with_encrypted_search_scheme("sse")
    .with_chunk_size(180)
    .add_documents(docs)
    .build_local(workers=2)
)

sse_cfg = PrivacyConfig(
    protocol=PrivacyProtocol.ENCRYPTED_SEARCH,
    backend=base_url,
    top_k=3,
    encrypted_search_scheme="sse",
)

sse_agent = SecureRAGAgent.from_config(
    sse_cfg,
    corpus=sse_corpus,
    llm=OllamaLLM(use_ollama=False),
)

sse_docs = sse_agent.retriever.retrieve("q3 risk vendor concentration", round_n=0)
print([d.doc_id for d in sse_docs[:3]])
print(sse_docs[0].text[:140])

In [ ]:
plugin = sse_corpus.extras["plugin"]
key = sse_corpus.extras["enc_key"]
encrypted_query = plugin.encrypt_query("q3 risk vendor concentration", key)

print("Encrypted query fields:", list(encrypted_query.keys()))
print("Encrypted term sample:", encrypted_query["enc_terms"][0][:20] + "...")
print("Any plaintext token leaked?", any(tok in str(encrypted_query) for tok in ["q3", "risk", "vendor"]))

## 2) Structured Scheme with Bigram Toggle

Structured retrieval can be explored with and without bigrams for ablation-style studies.

In [ ]:
structured_on = (
    CorpusBuilder(PrivacyProtocol.ENCRYPTED_SEARCH, backend_url=base_url)
    .with_encrypted_search_scheme("structured", structured_use_bigrams=True)
    .add_documents(docs)
    .build_local(workers=2)
)

structured_off = (
    CorpusBuilder(PrivacyProtocol.ENCRYPTED_SEARCH, backend_url=base_url)
    .with_encrypted_search_scheme("structured", structured_use_bigrams=False)
    .add_documents(docs)
    .build_local(workers=2)
)

print("bigrams ON:", structured_on.extras["plugin"].use_bigrams)
print("bigrams OFF:", structured_off.extras["plugin"].use_bigrams)

## 3) Version Guard Demonstration

Indexes marked with a legacy encrypted-search version should be rejected with a rebuild message.

In [ ]:
backend = create_backend(base_url)
sse_plugin = EncryptedSchemePlugin.get("sse")
legacy_key = "0123456789abcdef0123456789abcdef"

chunks = [
    {
        "doc_id": d.doc_id,
        "text": d.text,
        "metadata": d.metadata,
        "scheme_data": sse_plugin.prepare_chunk(d.text, legacy_key),
    }
    for d in docs
]

legacy_idx = backend.build_index(
    "EncryptedSearch",
    chunks,
    encrypted_search_scheme="sse",
    encrypted_search_version="sha256-v0",
)

try:
    backend.encrypted_search(
        legacy_idx["index_id"],
        sse_plugin.encrypt_query("q3 risk", legacy_key),
        3,
    )
except BackendError as exc:
    print("Expected guard triggered:")
    print(exc)

In [ ]:
# Cleanup local server process started in this tutorial.
sim_proc.terminate()
try:
    sim_proc.wait(timeout=2)
except subprocess.TimeoutExpired:
    sim_proc.kill()

print("sim_server stopped")